In [ ]:
from dataloading import *
from bpetokenizer import *
from CJPEDataset import *
from model_arch import *

In [ ]:
train_ds, dev_ds, test_ds = load_cjpe()
tokenizer = load_bpe_tokenizer()

train_dataset = CJPEDataset(train_ds)
dev_dataset = CJPEDataset(dev_ds)
test_dataset = CJPEDataset(test_ds)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

In [ ]:
class Phase4ClassificationModel(nn.Module):
    def __init__(self, base_model, num_classes=2, pad_token_id=0):
        super().__init__()
        self.base_model = base_model
        self.num_classes = num_classes
        self.pad_token_id = pad_token_id
        
        self.classification_head = nn.Linear(self.base_model.cfg.d_model, num_classes)

    def forward(self, idx):
        B, T = idx.shape

        x = self.base_model.tok_emb(idx)
        for block in self.base_model.blocks:
            x = block(x)
        hidden_states = self.base_model.ln_f(x)

        sequence_lengths = (idx != self.pad_token_id).sum(dim=1)
        last_token_indices = torch.clamp(sequence_lengths - 1, min=0)
        tail_output = hidden_states[torch.arange(B, device=idx.device), last_token_indices]
        logits = self.classification_head(tail_output)
        
        return logits

In [ ]:
import torch.optim

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
cfg = LMConfig(vocab_size=50260)
base_lm = FourthDecoderLM(cfg)
base_lm.load_state_dict(torch.load("mla.pt", map_location=device))

model = Phase4ClassificationModel(base_lm, num_classes=2, pad_token_id=0)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import f1_score

epochs = 3

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    
    for batch_idx, (b_inputs, b_labels) in enumerate(train_loader):
        b_inputs, b_labels = b_inputs.to(device), b_labels.to(device)
        
        optimizer.zero_grad()
        logits = model(b_inputs)
        loss = criterion(logits, b_labels)
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
            
    model.eval()
    all_preds = []
    all_targets = []
    val_loss = 0
    
    with torch.no_grad():
        for b_inputs, b_labels in dev_loader:
            b_inputs, b_labels = b_inputs.to(device), b_labels.to(device)
            
            logits = model(b_inputs)
            loss = criterion(logits, b_labels)
            val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(b_labels.cpu().numpy())
            
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = val_loss / len(dev_loader)
    
    print("Train Loss:", avg_train_loss, "Val Loss:", avg_val_loss, "Val Macro-F1:", macro_f1)

    torch.save(model.state_dict(), "checkpoint" + str(epoch) + ".pt")